# 02 – Alias reliability check (Step 3)
Compare the hand-built alias tables in `manifest/annotations.csv` with a rule-based merge
(title + surname, surname only, full-name containment) on the pilot stories P1–P5.
Only text **before** the reveal paragraph is used (CLAUDE.md rule 5).

Scoring: pairwise precision/recall over the gold alias forms that occur on their own in the prefix
(`scorable`; e.g. "Windibank" only ever appears inside "Mr. Windibank", so it is not scored).
A scorable form the extractor never finds counts as an unlinked singleton, so it lowers recall.
Pairs with names outside the gold table are not scored, so wrong merges with other characters
are only visible in the last cell. `culprit_share` = share of the principal culprit's scorable forms
that land in one rule cluster.

In [ ]:
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd, spacy
from src.aliases import run_pilot, standalone_forms, prefix_paragraphs
nlp = spacy.load("en_core_web_sm")

In [ ]:
summary, details = run_pilot(nlp)
summary.round(3)

In [ ]:
# Micro-averaged over all pilot pairs
tp, fp, fn = summary[["tp", "fp", "fn"]].sum()
print(f"pairs: precision {tp / (tp + fp):.3f}, recall {tp / (tp + fn):.3f}")
print(f"scorable gold forms extracted: {summary.extracted.sum()} / {summary.scorable.sum()} "
      f"({summary.gold_forms.sum()} gold forms in total)")

In [ ]:
# Error analysis: how the rules grouped each gold character's forms
manifest = pd.read_csv("manifest/annotations.csv").set_index("story_id")
for sid, d in details.items():
    scorable = standalone_forms(d["gold"], prefix_paragraphs(sid, int(manifest.loc[sid].reveal_para_idx)))
    pred = {s.lower(): c for s, c in d["pred"].items()}
    print(f"== {sid}  principal-culprit forms outside its main cluster: {d['culprit_missed']}")
    for char, forms in d["gold"].items():
        groups = {}
        for f in forms:
            key = (f"cluster {pred[f.lower()]}" if f.lower() in pred else "not extracted") if f in scorable \
                else "only inside a longer form"
            groups.setdefault(key, []).append(f)
        print(f"  {char}: " + " | ".join(f"{k}: {v}" for k, v in groups.items()))

In [ ]:
# Rule clusters with more than one surface form (includes characters outside the gold tables)
for sid, d in details.items():
    clusters = {}
    for s, c in d["pred"].items():
        clusters.setdefault(c, []).append(s)
    print(f"== {sid}")
    for forms in clusters.values():
        if len(forms) > 1:
            print("  ", sorted(forms))

In [ ]:
os.makedirs("outputs", exist_ok=True)
summary.to_csv("outputs/alias_check_pilot.csv", index=False)

## Merge variants: pseudonym links stated only at/after the reveal
`manifest/alias_links.csv` records, for each pseudonym/misnaming, the paragraph that states the link.
*Prefix-only* hand merges drop links whose paragraph is ≥ the reveal (CLAUDE.md rule 5 exception).
For the principal culprit we compare three rosters: rules only, rules + prefix-only hand merges,
rules + all hand merges. `freq_rank` is the culprit's rank by prefix mention count among all
extracted characters (detective, narrator and NER noise included), i.e. the mention-frequency baseline.

In [ ]:
from src.aliases import compare_merge_variants
visibility, variant_scores = compare_merge_variants(nlp)
visibility

In [ ]:
# name-string recall leaves out pairs that need a pseudonym/misnaming link (fn_link):
# those can only come from the hand table / alias_links.csv, never from name-string rules
for g, d in variant_scores.groupby("gold", sort=False):
    t = d[["tp", "fp", "fn", "fn_not_extracted", "fn_link", "fn_rules"]].sum()
    print(f"rules vs {g}: precision {t.tp / (t.tp + t.fp):.3f}, recall {t.tp / (t.tp + t.fn):.3f}, "
          f"name-string recall {t.tp / (t.tp + t.fn_not_extracted + t.fn_rules):.3f}; "
          f"missed pairs: not extracted {t.fn_not_extracted}, link {t.fn_link}, rules {t.fn_rules}")
variant_scores.round(3)

In [ ]:
visibility.to_csv("outputs/alias_merge_variants_visibility.csv", index=False)
variant_scores.to_csv("outputs/alias_merge_variants_scores.csv", index=False)